# Notebook para acesso a buckets da AWS
Autor: Suzane Carol de Lima  
Entrega: --
## Objetivo:
Notebook para auxiliar nos testes de acesso a buckets da AWS via boto3


## Importações Necessárias

In [15]:
import boto3
import os
import boto3
from botocore.exceptions import BotoCoreError, NoCredentialsError, ClientError
from getpass import getpass
import urllib.parse
import json

## Funções Necessárias

In [16]:
def load_config(path: str) -> dict:
    """
    Loads and returns the content of a JSON configuration file.

    Args:
        path (str): Path to the JSON file.

    Returns:
        dict: The content of the JSON file as a dictionary.

    Raises:
        FileNotFoundError: If the file does not exist.
        json.JSONDecodeError: If the file is not a valid JSON.
    """
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {path}")
    except json.JSONDecodeError:
        raise json.JSONDecodeError(f"Invalid JSON format in file: {path}", doc="", pos=0)



def create_s3_client(aws_access_key_id: str, aws_secret_access_key: str) -> boto3.client:
    """
    Creates and returns an S3 client using the provided AWS credentials.

    Args:
        aws_access_key_id (str): The AWS access key ID.
        aws_secret_access_key (str): The AWS secret access key.

    Returns:
        boto3.client: An S3 client object.

    Raises:
        ValueError: If credentials are missing.
        BotoCoreError, NoCredentialsError, ClientError: If client creation fails.
    """
    if not aws_access_key_id or not aws_secret_access_key:
        raise ValueError("AWS credentials must be provided.")
    try:
        return boto3.client(
            's3',
            aws_access_key_id=aws_access_key_id,
            aws_secret_access_key=aws_secret_access_key
        )
    except (BotoCoreError, NoCredentialsError, ClientError) as e:
        raise RuntimeError(f"Failed to create S3 client: {str(e)}")


def download_s3_file(s3_client: boto3.client, bucket_name: str, object_key: str, local_path: str) -> None:
    """
    Downloads a file from an S3 bucket to a local path.

    Args:
        s3_client (boto3.client): Authenticated S3 client.
        bucket_name (str): Name of the S3 bucket.
        object_key (str): Path to the file in the S3 bucket.
        local_path (str): Local path where the file will be saved.

    Returns:
        None

    Raises:
        RuntimeError: If the download fails.
    """
    try:
        s3_client.download_file(bucket_name, object_key, local_path)
    except (BotoCoreError, ClientError) as e:
        raise RuntimeError(f"Failed to download S3 file: {str(e)}")

def upload_s3_file(s3_client: boto3.client, local_path: str, bucket_name: str, s3_path: str) -> None:
    """
    Uploads a local file to an S3 bucket.

    Args:
        s3_client (boto3.client): Authenticated S3 client.
        local_path (str): Local file path.
        bucket_name (str): Name of the S3 bucket.
        s3_path (str): Destination path in the S3 bucket.

    Returns:
        None

    Raises:
        RuntimeError: If the upload fails.
    """
    try:
        s3_client.upload_file(local_path, bucket_name, s3_path)
    except (BotoCoreError, ClientError) as e:
        raise RuntimeError(f"Failed to upload file to S3: {str(e)}")

def list_s3_files(s3_client: boto3.client, bucket_name: str, folder_path: str) -> list:
    """
    Lists all files in a specific folder of an S3 bucket.

    Args:
        s3_client (boto3.client): Authenticated S3 client.
        bucket_name (str): Name of the S3 bucket.
        folder_path (str): Path of the folder within the bucket (prefix).

    Returns:
        list: List of full file paths (keys) found in the specified folder.

    Raises:
        RuntimeError: If listing files fails.
    """
    try:
        response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=folder_path)
        return [obj['Key'] for obj in response.get('Contents', [])]
    except (BotoCoreError, ClientError) as e:
        raise RuntimeError(f"Failed to list files in S3: {str(e)}")

def list_s3_folders(s3_client: boto3.client, bucket_name: str) -> list:
    """
    Lists all folders (prefixes) in an S3 bucket.

    Args:
        s3_client (boto3.client): Authenticated S3 client.
        bucket_name (str): Name of the S3 bucket.

    Returns:
        list: List of folder paths (prefixes) found in the bucket.

    Raises:
        RuntimeError: If listing folders fails.
    """
    try:
        response = s3_client.list_objects_v2(Bucket=bucket_name, Delimiter='/')
        return [prefix['Prefix'] for prefix in response.get('CommonPrefixes', [])]
    except (BotoCoreError, ClientError) as e:
        raise RuntimeError(f"Failed to list folders in S3: {str(e)}")


    
def download_s3_folder(s3: boto3.client, bucket, prefix, local_dir):
    """
    Downloads all files from a specific folder (prefix) in an S3 bucket to a local directory,
    preserving the subfolder structure. Prints the name of each subfolder found during the process.

    Args:
        s3 (boto3.client): Authenticated S3 client.
        bucket (str): Name of the S3 bucket.
        prefix (str): S3 folder prefix to download.
        local_dir (str): Local path where files will be saved.

    Returns:
        None
    """
    paginator = s3.get_paginator('list_objects_v2')
    pastas_printadas = set()
    try:
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get('Contents', []):
                s3_key = obj['Key']
                if s3_key.endswith('/'):
                    continue
                relativo = s3_key[len(prefix):]
                if '/' in relativo:
                    pasta = relativo.split('/')[0]
                    if pasta and pasta not in pastas_printadas:
                        print(pasta)
                        pastas_printadas.add(pasta)
                local_path = os.path.join(local_dir, relativo)
                os.makedirs(os.path.dirname(local_path), exist_ok=True)
                try:
                    s3.download_file(bucket, s3_key, local_path)
                except (BotoCoreError, ClientError) as e:
                    print(f'Erro ao baixar {s3_key}: {e}')
    except (BotoCoreError, ClientError) as e:
        print(f'Erro ao listar objetos: {e}')

## Declaração de Constantes (se tiver)

In [17]:
BUCKET_NAME = 'analise-dados'
config = load_config('configuracao_aws/config_onboard.json')
AWS_ACCESS_KEY_ID = config['analise-dados']['aws_access_key_id']
AWS_SECRET_ACCESS_KEY = config['analise-dados']['aws_secret_access_key']

## Criação do client do S3

In [18]:
s3_client = create_s3_client(AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)

## Listando os arquivos de uma pasta

In [19]:
folder_path = 'projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/images/'
list_s3_files(s3_client, BUCKET_NAME, folder_path)

['projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/images/test/20240324-231713-ic_COMco24_025.jpg',
 'projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/images/test/20240324-231746-ic_COMco24_025.jpg',
 'projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/images/test/20240324-231749-ic_COMco24_025.jpg',
 'projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/images/test/20240324-231849-ic_COMco24_025.jpg',
 'projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/images/test/20240324-232101-ic_COMco24_025.jpg',
 'projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/images/test/20240325-012707-ic_COMco24_025.jpg',
 'projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/images/test/20240325-012854-ic_COMco24_025.jpg',
 'projeto-ia-submarina/ia-frente-a

## Listando as pastas do bucket

In [20]:
list_s3_folders(s3_client, BUCKET_NAME)

['inventario-diario/',
 'inventario/',
 'projeto-ia-submarina/',
 'projeto-manntis/',
 'sandbox/']

## Download de um arquivo

In [21]:
object_key='projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/images/train/SUB9srr24_064__20240327-185116.jpg'
local_path= 'teste_aws/SUB9srr24_064__20240327-185116.jpg'
download_s3_file(s3_client, BUCKET_NAME, object_key, local_path)

## Download de todos os arquivos de uma pasta

In [14]:
prefix = 'projeto-ia-submarina/ia-frente-ambiental/Datasets/coral_images_lote_semOverlayRevisado/labels/test/'
local_dir = 'teste_aws/'
download_s3_folder(s3_client, BUCKET_NAME, prefix, local_dir)

## Enviando arquivo para o bucket

In [ ]:
local_path='caminho do arquivo local'
s3_path = 'caminho do arquivo que será adicionado na pasta no bucket'
upload_s3_file(s3_client, local_path, BUCKET_NAME, s3_path)